# CodeTune v2 — 07 DPO v2 Data Generation

**Environment**: Google Colab A100 80GB  
**Model**: `Michlitt/codetune-v2-sft-C`（生成候选，bf16，~18GB VRAM）  
**Output**: Drive `dpo_v2_pairs/dpo_pairs_v2.jsonl` + `dpo_pairs_v2_val.jsonl`

## 相比 v1 的改进

| 维度 | dpo_v1 数据（06 notebook） | dpo_v2 数据（本 notebook） |
|------|---------------------------|---------------------------|
| 生成模型 | sft_C | **sft_C**（不变，能力更强） |
| 候选数量 | 4 个/题 | **8 个/题**（增加 pairs 命中率） |
| 总 pairs 目标 | ~147 对 | **500+ 对** |
| Targeted pairs | HumanEval 8题 mutated bug | **HE/108、HE/128 精准构造 + 执行验证** |
| 退步修复 pairs | 无 | **dpo_v1 退步的 ~6 题：canonical chosen vs dpo_v1 eval 中的错误输出 rejected** |

> **为什么不用 dpo_v1 生成？**  
> dpo_v1 整体 pass@1（72.6%）低于 sft_C（76.2%），生成质量更差。  
> DPO chosen 的质量上限决定训练信号上限，应始终用能力更强的 sft_C 生成。  
> dpo_v1 的价值仅在于提供退步题目的 rejected 样本（从其评估结果中直接读取）。

## 运行顺序

| Cell | 任务 | 预计时间 |
|------|------|----------|
| 1 | GPU 检查 | 即时 |
| 2 | 安装依赖 | ~3 min |
| 3 | HF 登录 + Drive 挂载 | ~1 min |
| 4 | 配置 | 即时 |
| 5 | 下载 sft_C + 加载模型 + 工具函数 | ~5 min |
| 6 | 加载 HumanEval | 即时 |
| 7 | 主循环：8温度 batch 推理 + 并行测试 | ~25 min |
| 8 | Targeted pairs（HE/108、HE/128） | 即时 |
| 9 | 退步修复 pairs（从 dpo_v1 eval JSON 读取 rejected） | 即时 |
| 10 | 合并、去重、统计 | 即时 |
| 11 | 分割、保存 + Drive 备份 | 即时 |
| 12 | 上传到 HF Hub dataset | ~1 min |
| 13 | 质量预览 | 即时 |

In [ ]:
# Cell 1 — GPU 检查
import subprocess, torch
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", r.stdout.strip())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, "
      f"BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 2 — 安装（unsloth 必须最先安装）
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
!pip install datasets huggingface-hub -q

import sys, types
from importlib.machinery import ModuleSpec

_STUBS = [
    "llm_blender", "llm_blender.blender", "llm_blender.blender.blender_utils",
    "llm_blender.pair_ranker", "weave", "weave.trace",
    "weave.trace.context", "weave.trace.context.weave_client_context",
]
for _name in _STUBS:
    m = types.ModuleType(_name)
    m.__spec__ = ModuleSpec(_name, None)
    m.__file__ = None
    m.__path__ = []
    m.__package__ = _name.split(".")[0]
    sys.modules[_name] = m

sys.modules["weave"].EvaluationLogger = type("EvaluationLogger", (), {})
sys.modules["weave.trace.context.weave_client_context"].get_weave_client = lambda: None

print("Done.")

In [ ]:
# Cell 3 — HF 登录 + Drive 挂载
from huggingface_hub import login
from google.colab import drive
import os

HF_TOKEN    = "YOUR_HF_TOKEN"
HF_USERNAME = "Michlitt"

login(token=HF_TOKEN)
print("Logged in as", HF_USERNAME)

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/codetune"
print(f"Drive mounted. DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
# Cell 4 — 配置
from pathlib import Path

SFT_C_CHECKPOINT = f"{HF_USERNAME}/codetune-v2-sft-C"  # 生成候选用（能力更强）
CHECKPOINT_DIR   = Path("sft_C")

OUTPUT_TRAIN = Path("data/processed/dpo_pairs_v2.jsonl")
OUTPUT_VAL   = Path("data/processed/dpo_pairs_v2_val.jsonl")
DRIVE_BACKUP = Path(DRIVE_ROOT) / "dpo_v2_pairs"

# dpo_v1 评估结果（用于退步修复 pairs 的 rejected 来源）
# 文件在 Drive eval_results 目录下，由 05_eval_colab.ipynb 生成
DRIVE_EVAL_RESULTS = Path(DRIVE_ROOT) / "eval_results"
DPO_V1_EVAL_JSON   = DRIVE_EVAL_RESULTS / "humaneval_dpo_v1.json"
SFT_C_EVAL_JSON    = DRIVE_EVAL_RESULTS / "humaneval_sft_C.json"

# 8个温度候选：覆盖更广，增加 pairs 命中率
TEMPERATURES   = [0.2, 0.4, 0.6, 0.8, 1.0, 1.1, 1.2, 1.3]
MAX_NEW_TOKENS = 512
TEST_TIMEOUT   = 10
VAL_RATIO      = 0.1
SEED           = 42

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Complete the given function. "
    "Write only the function body — no extra explanation, no test code."
)

print("Config loaded.")
print(f"  Generation model  : {SFT_C_CHECKPOINT}")
print(f"  Temperatures      : {TEMPERATURES}  ({len(TEMPERATURES)} candidates/problem)")
print(f"  dpo_v1 eval JSON  : {DPO_V1_EVAL_JSON}")
print(f"  Output train      : {OUTPUT_TRAIN}")

In [ ]:
# Cell 5 — 下载 sft_C + 加载模型（bf16）+ 工具函数
import sys, subprocess, tempfile, torch
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

# ── 下载 sft_C ───────────────────────────────────────────────────────────
CHECKPOINT_DIR.mkdir(exist_ok=True)
if not (CHECKPOINT_DIR / "adapter_config.json").exists():
    print(f"Downloading {SFT_C_CHECKPOINT} ...")
    snapshot_download(
        repo_id=SFT_C_CHECKPOINT,
        local_dir=str(CHECKPOINT_DIR),
        repo_type="model",
    )
    print("Download done.")
else:
    print(f"Already downloaded: {CHECKPOINT_DIR}")

# ── 加载 sft_C（A100 80GB，bf16） ────────────────────────────────────────
print("Loading sft_C model (bf16) ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(CHECKPOINT_DIR),
    max_seq_length=1536,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
used = torch.cuda.memory_allocated(0) / 1e9
print(f"Model ready. VRAM used: {used:.1f} GB")


# ── 工具函数 ──────────────────────────────────────────────────────────────
def clean_completion(raw: str, prompt: str) -> str:
    """Normalize raw model output to 4-space-indented function body."""
    import re
    text = raw.rstrip()
    # Strip think blocks
    text = re.sub(r"<think>[\s\S]*?</think>\s*", "", text)
    # Strip markdown fences
    text = re.sub(r"^```(?:python)?\s*
", "", text)
    text = re.sub(r"
?```\s*$", "", text)
    text = text.strip("
")
    # Strip ": " line prefix (Qwen3 generation artifact)
    text = re.sub(r"^( *):[ ]?", r"", text, flags=re.MULTILINE)
    # Strip echoed function header + docstring
    fn_match = re.search(r"^def\s+(\w+)", prompt, re.MULTILINE)
    if fn_match:
        fn = re.escape(fn_match.group(1))
        hdr = re.search(rf"def\s+{fn}\s*\(.*?\).*?:\s*
", text, re.DOTALL)
        if hdr:
            after = text[hdr.end():]
            doc = re.match(r'\s*"""[\s\S]*?"""\s*
', after)
            text = after[doc.end():] if doc else after
    text = text.strip("
")
    # Normalize indentation to 4-space base (preserve relative indent)
    lines = text.splitlines()
    non_empty = [l for l in lines if l.strip()]
    if not non_empty:
        return text
    min_indent = min(len(l) - len(l.lstrip()) for l in non_empty)
    if min_indent != 4:
        shift = 4 - min_indent
        result = []
        for l in lines:
            if not l.strip():
                result.append("")
            elif shift > 0:
                result.append(" " * shift + l)
            else:
                result.append(l[-shift:])
        text = "
".join(result)
    return text


def format_prompt(problem_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Complete this Python function:\n\n{problem_prompt}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def batch_generate(formatted_prompts: list[str], temperature: float, batch_size: int = 16) -> list[str]:
    """对所有 prompts 以同一 temperature 做 batch 推理，返回与输入等长的 completion 列表。"""
    do_sample = temperature > 0.0
    results = []
    for i in range(0, len(formatted_prompts), batch_size):
        chunk = formatted_prompts[i : i + batch_size]
        enc = tokenizer(
            text=chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            padding=True,
        ).to(model.device)
        kwargs = dict(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            kwargs["temperature"] = temperature
            kwargs["top_p"] = 0.95
        with torch.no_grad():
            outs = model.generate(**kwargs)
        prompt_len = enc["input_ids"].shape[1]
        for out in outs:
            results.append(
                tokenizer.decode(out[prompt_len:], skip_special_tokens=True).rstrip()
            )
    return results


def run_tests(problem_prompt: str, completion: str,
              test_code: str, entry_point: str) -> dict:
    full_code = (
        problem_prompt + completion + "\n\n"
        + test_code + f"\n\ncheck({entry_point})\n"
    )
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8"
    ) as f:
        f.write(full_code)
        tmp = f.name
    try:
        r = subprocess.run(
            [sys.executable, tmp],
            capture_output=True, text=True, timeout=TEST_TIMEOUT,
        )
        return {"passed": r.returncode == 0, "stderr": r.stderr[:300]}
    except subprocess.TimeoutExpired:
        return {"passed": False, "stderr": "TIMEOUT"}
    except Exception as e:
        return {"passed": False, "stderr": str(e)}
    finally:
        Path(tmp).unlink(missing_ok=True)


print("Helper functions defined.")

In [ ]:
# Cell 6 — 加载 HumanEval
from datasets import load_dataset

he_ds = load_dataset("openai/openai_humaneval", split="test")
problems = list(he_ds)
print(f"HumanEval: {len(problems)} problems")

# 预处理所有 prompt
formatted = [format_prompt(p["prompt"]) for p in problems]
print(f"Formatted {len(formatted)} prompts.")

In [ ]:
# Cell 7 — 主循环：8温度 batch 推理 + 并行执行测试
# 预计耗时：~20 min 生成 + ~1 min 测试
#
# 扩大 pairs 的关键：每题允许多对（所有通过候选 × 所有失败候选的笛卡尔积），
# 但限制每题最多 MAX_PAIRS_PER_PROBLEM 对，避免高频题目主导训练。
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import itertools

MAX_PAIRS_PER_PROBLEM = 4   # 每题最多取多少对（chosen, rejected 组合）

n = len(problems)

# ── Step 1: 8个温度 batch 生成 ────────────────────────────────────────────
all_candidates = {}  # {temperature: [completion_0, ..., completion_163]}
for t in TEMPERATURES:
    print(f"Generating temperature={t} ...")
    all_candidates[t] = batch_generate(formatted, temperature=t, batch_size=16)
print(f"\nGeneration done. {len(TEMPERATURES)} passes × {n} problems = {len(TEMPERATURES)*n} completions total.")

# ── Step 2: 并行执行测试 ──────────────────────────────────────────────────
NUM_WORKERS = 32

def test_problem(i):
    p           = problems[i]
    prompt      = p["prompt"]
    test_code   = p["test"]
    entry_point = p["entry_point"]
    canonical   = p["canonical_solution"]
    candidates  = [clean_completion(all_candidates[t][i], prompt) for t in TEMPERATURES]

    results  = [run_tests(prompt, c, test_code, entry_point) for c in candidates]
    passing  = [c for c, r in zip(candidates, results) if r["passed"]]
    failing  = [c for c, r in zip(candidates, results) if not r["passed"]]
    return i, candidates, passing, failing, canonical, p["task_id"]

exec_pairs = []
sft_pairs  = []
skipped_all_pass   = 0
skipped_canon_fail = 0

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(test_problem, i): i for i in range(n)}
    for fut in tqdm(as_completed(futures), total=n, desc="Testing & pairing", unit="problem"):
        i, candidates, passing, failing, canonical, task_id = fut.result()

        if passing and failing:
            # 所有通过×失败组合，按 (chosen低温, rejected高温) 排序取前 N 对
            # passing 已按温度从低到高排列（最保守优先），failing 从高到低（最发散优先）
            combos = list(itertools.product(passing, failing))
            for chosen, rejected in combos[:MAX_PAIRS_PER_PROBLEM]:
                if chosen.strip() != rejected.strip():
                    exec_pairs.append({
                        "prompt":   formatted[i],
                        "chosen":   chosen,
                        "rejected": rejected,
                        "source":   "execution_driven",
                        "task_id":  task_id,
                    })
        elif not passing:
            # 全部失败：用 canonical 做 chosen，多个失败候选都可作为 rejected
            r_canon = run_tests(problems[i]["prompt"], canonical,
                                problems[i]["test"], problems[i]["entry_point"])
            if r_canon["passed"]:
                for rejected in failing[:MAX_PAIRS_PER_PROBLEM]:
                    if rejected.strip() != canonical.strip():
                        sft_pairs.append({
                            "prompt":   formatted[i],
                            "chosen":   canonical,
                            "rejected": rejected,
                            "source":   "sft_failure",
                            "task_id":  task_id,
                        })
            else:
                skipped_canon_fail += 1
        else:
            skipped_all_pass += 1

print(f"\n执行驱动对数  (execution_driven): {len(exec_pairs)}")
print(f"SFT失败兜底对数 (sft_failure)    : {len(sft_pairs)}")
print(f"跳过（全通过）                   : {skipped_all_pass}")
print(f"跳过（canonical也失败）          : {skipped_canon_fail}")
print(f"理论上限                         : {n} × {MAX_PAIRS_PER_PROBLEM} = {n * MAX_PAIRS_PER_PROBLEM} 对")

In [ ]:
# Cell 8 — Targeted pairs：HE/108 (intersection) + HE/128 (tri)
# 这两题在所有版本中持续失败，手工构造高质量 chosen/rejected

# 从 HumanEval 中取出对应题目
he_by_id = {p["task_id"]: p for p in problems}

TARGETED_PAIRS = [
    # ── HE/108: intersection ────────────────────────────────────────────
    # 模型常见错误：忽略空集合交集为空，或对区间端点处理不当
    {
        "task_id": "HumanEval/108",
        "chosen_body": (
            "    l = max(interval1[0], interval2[0])\n"
            "    r = min(interval1[1], interval2[1])\n"
            "    if l > r:\n"
            "        return \"NO\"\n"
            "    length = r - l\n"
            "    if length < 2:\n"
            "        return \"NO\"\n"
            "    for i in range(2, length + 1):\n"
            "        if length % i == 0:\n"
            "            return \"NO\"\n"
            "    return \"YES\"\n"
        ),
        "rejected_body": (
            "    l = max(interval1[0], interval2[0])\n"
            "    r = min(interval1[1], interval2[1])\n"
            "    if l >= r:\n"  # 错误：应为 l > r
            "        return \"NO\"\n"
            "    length = r - l\n"
            "    for i in range(2, length):\n"  # 错误：range 少了 length 本身
            "        if length % i == 0:\n"
            "            return \"NO\"\n"
            "    return \"YES\"\n"
        ),
        "bug_type": "off_by_one_in_range_and_wrong_empty_check",
    },
    # ── HE/128: tri ─────────────────────────────────────────────────────
    # 模型常见错误：混淆奇偶递推公式，或 1-indexed vs 0-indexed
    {
        "task_id": "HumanEval/128",
        "chosen_body": (
            "    if n == 0:\n"
            "        return [1]\n"
            "    my_tri = [1, 3]\n"
            "    for i in range(2, n + 1):\n"
            "        if i % 2 == 0:\n"
            "            my_tri.append(1 + i / 2)\n"
            "        else:\n"
            "            my_tri.append(my_tri[i - 1] + my_tri[i - 2] + (i + 3) / 2)\n"
            "    return my_tri[:n + 1]\n"
        ),
        "rejected_body": (
            "    if n == 0:\n"
            "        return [1]\n"
            "    my_tri = [1, 3]\n"
            "    for i in range(2, n + 1):\n"
            "        if i % 2 == 0:\n"
            "            my_tri.append(1 + i // 2)\n"  # 错误：整除而非浮点除
            "        else:\n"
            "            my_tri.append(my_tri[i - 1] + my_tri[i - 2] + (i + 3) // 2)\n"  # 同上
            "    return my_tri\n"  # 错误：未切片
        ),
        "bug_type": "integer_division_vs_float_and_missing_slice",
    },
]

targeted_pairs = []
for spec in TARGETED_PAIRS:
    tid = spec["task_id"]
    p   = he_by_id.get(tid)
    if not p:
        print(f"WARNING: {tid} not found in HumanEval")
        continue

    prompt_text = format_prompt(p["prompt"])
    chosen      = spec["chosen_body"]
    rejected    = spec["rejected_body"]

    # 验证 chosen 确实通过测试
    r = run_tests(p["prompt"], chosen, p["test"], p["entry_point"])
    if not r["passed"]:
        print(f"WARNING: {tid} chosen body FAILS test! stderr: {r['stderr'][:200]}")
        print("Skipping this targeted pair.")
        continue

    targeted_pairs.append({
        "prompt":   prompt_text,
        "chosen":   chosen,
        "rejected": rejected,
        "source":   "targeted_v2",
        "task_id":  tid,
        "bug_type": spec["bug_type"],
    })
    print(f"  {tid}: chosen PASS ✓  → targeted pair added")

print(f"\nTargeted pairs added: {len(targeted_pairs)}")

In [ ]:
# Cell 9 — 退步修复 pairs
# 从 dpo_v1 评估 JSON 中读取退步题目的错误输出作为 rejected，
# 不需要重新加载 dpo_v1 模型。
#
# 退步定义：sft_C 通过 但 dpo_v1 失败 的题目。
# chosen = canonical_solution（执行验证通过）
# rejected = dpo_v1 eval 中保存的错误 completion

import json

regression_pairs = []

if not DPO_V1_EVAL_JSON.exists():
    print(f"WARNING: {DPO_V1_EVAL_JSON} 不存在，跳过退步修复 pairs。")
    print("请确认 05_eval_colab.ipynb 已运行并保存了 humaneval_dpo_v1.json。")
elif not SFT_C_EVAL_JSON.exists():
    print(f"WARNING: {SFT_C_EVAL_JSON} 不存在，跳过退步修复 pairs。")
else:
    dpo_v1_eval = json.loads(DPO_V1_EVAL_JSON.read_text(encoding="utf-8"))
    sft_c_eval  = json.loads(SFT_C_EVAL_JSON.read_text(encoding="utf-8"))

    dpo_v1_per = dpo_v1_eval.get("per_problem", {})
    sft_c_per  = sft_c_eval.get("per_problem", {})

    # 找出退步题目：sft_C 通过 且 dpo_v1 失败
    regression_tids = [
        tid for tid in sft_c_per
        if sft_c_per[tid].get("passed") and not dpo_v1_per.get(tid, {}).get("passed")
    ]
    print(f"退步题目数: {len(regression_tids)}")
    print(f"  {regression_tids}")

    he_by_id = {p["task_id"]: p for p in problems}

    for tid in regression_tids:
        p = he_by_id.get(tid)
        if not p:
            continue

        rejected = dpo_v1_per[tid].get("completion", "")
        if not rejected.strip():
            print(f"  {tid}: dpo_v1 completion 为空，跳过")
            continue

        canonical = p["canonical_solution"]

        # 验证 canonical 通过测试
        r_chosen = run_tests(p["prompt"], canonical, p["test"], p["entry_point"])
        if not r_chosen["passed"]:
            print(f"  {tid}: canonical 未通过测试，跳过")
            continue

        # 验证 dpo_v1 completion 确实失败
        r_rejected = run_tests(p["prompt"], rejected, p["test"], p["entry_point"])
        if r_rejected["passed"]:
            print(f"  {tid}: dpo_v1 completion 意外通过测试（eval 截断导致？），跳过")
            continue

        regression_pairs.append({
            "prompt":   format_prompt(p["prompt"]),
            "chosen":   canonical,
            "rejected": rejected,
            "source":   "regression_fix",
            "task_id":  tid,
        })
        print(f"  {tid}: chosen PASS ✓  rejected FAIL ✓ → regression pair added")

    print(f"\n退步修复 pairs: {len(regression_pairs)}")

In [ ]:
# Cell 10 — 合并、去重、统计
# 去重逻辑：同一 (task_id, chosen, rejected) 三元组视为重复，其余保留。
# 不再强制每题只保留一对——多对可以增强训练信号。
# targeted_v2 / regression_fix 仍保持优先（不会被 exec_driven 覆盖）。
from collections import Counter

priority = {"targeted_v2": 0, "regression_fix": 1, "execution_driven": 2, "sft_failure": 3}

all_pairs = targeted_pairs + regression_pairs + exec_pairs + sft_pairs

# 按 (task_id, chosen[:100], rejected[:100]) 去重，保留优先级高的
seen: dict[tuple, dict] = {}
for p in all_pairs:
    key = (p["task_id"], p["chosen"][:100], p["rejected"][:100])
    if key not in seen or priority[p["source"]] < priority[seen[key]["source"]]:
        seen[key] = p

deduped = list(seen.values())
removed = len(all_pairs) - len(deduped)

print(f"总对数（合并前）: {len(all_pairs)}")
print(f"去重后           : {len(deduped)}  （移除 {removed} 条完全相同对）")
print()

src_counts = Counter(p["source"] for p in deduped)
for src, cnt in src_counts.most_common():
    print(f"  {src:<35} {cnt}")

# 按 task_id 统计每题的 pair 数量分布
task_pair_counts = Counter(p["task_id"] for p in deduped)
avg = sum(task_pair_counts.values()) / len(task_pair_counts) if task_pair_counts else 0
print(f"\n覆盖题目数: {len(task_pair_counts)}/164")
print(f"每题平均 pairs: {avg:.1f}  (max={max(task_pair_counts.values(), default=0)})")

In [ ]:
# Cell 11 — 分割、保存 + Drive 备份
import json, random, shutil

random.seed(SEED)
random.shuffle(deduped)

split       = max(1, int(len(deduped) * (1 - VAL_RATIO)))
train_pairs = deduped[:split]
val_pairs   = deduped[split:]

OUTPUT_TRAIN.parent.mkdir(parents=True, exist_ok=True)
for path, data in [(OUTPUT_TRAIN, train_pairs), (OUTPUT_VAL, val_pairs)]:
    path.write_text(
        "\n".join(json.dumps(p, ensure_ascii=False) for p in data) + "\n",
        encoding="utf-8",
    )
    print(f"Saved: {path}  ({len(data)} pairs)")

DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
for path in [OUTPUT_TRAIN, OUTPUT_VAL]:
    shutil.copy(path, DRIVE_BACKUP / path.name)
    print(f"Drive backup: {DRIVE_BACKUP / path.name}")

print(f"\n合计: train={len(train_pairs)}  val={len(val_pairs)}")

In [ ]:
# Cell 12 — 上传到 HF Hub dataset
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
DATASET_REPO = f"{HF_USERNAME}/codetune-v2-sft"

for path in [OUTPUT_TRAIN, OUTPUT_VAL]:
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=DATASET_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded: {path.name} → {DATASET_REPO}")

print(f"\nDone. Dataset: https://huggingface.co/datasets/{DATASET_REPO}")

In [ ]:
# Cell 13 — 质量预览（随机打印 3 对，展示 chosen/rejected 对比）
import random

samples = random.sample(deduped, min(3, len(deduped)))

for i, pair in enumerate(samples):
    print("=" * 65)
    print(f"[{i+1}] task_id={pair['task_id']}  source={pair['source']}")

    lines = pair["prompt"].split("\n")
    start = next(
        (j for j, l in enumerate(lines) if "Complete this Python function" in l), 0
    )
    print("PROMPT:")
    print("\n".join(lines[start : start + 6]))
    print()
    print("CHOSEN (300 chars):")
    print(pair["chosen"][:300])
    print()
    print("REJECTED (300 chars):")
    print(pair["rejected"][:300])
    print()